In [ ]:
# 1. Install System FFmpeg (Crucial for video encoding)
!apt-get update && apt-get install -y ffmpeg

# 2. Install Python bindings
#!pip install imageio[ffmpeg] imageio-ffmpeg

In [ ]:
# ---------------------------------------------------------------------------
# CELL 1: CONFIGURATION
# ---------------------------------------------------------------------------
import os
import sys

# --- 1. Hardcoded Dataset Path ---
# This matches the dataset slug you used in the deploy script: "sage-pyapi-code"
PYAPI_DATASET_NAME = "sage-pyapi-code"
PYAPI_SOURCE_DIR = f"/kaggle/input/{PYAPI_DATASET_NAME}"

# Quick validation to ensure it's mounted
if not os.path.exists(os.path.join(PYAPI_SOURCE_DIR, "main.py")):
    print(f"❌ Error: 'main.py' not found in {PYAPI_SOURCE_DIR}")
    print(f"   Did you name the dataset '{PYAPI_DATASET_NAME}'? Check the right sidebar.")
    sys.exit(1)

print(f"✅ PyAPI Source Configured: {PYAPI_SOURCE_DIR}")

# --- 2. Zrok Token Setup ---
ZROK_TOKEN_PATH = "/kaggle/input/sage-zrok-token/.zrok_api_key"
zrok_token = None

if os.path.isfile(ZROK_TOKEN_PATH):
    with open(ZROK_TOKEN_PATH, "r", encoding="utf-8", errors="ignore") as f:
        zrok_token = f.read().strip()

# Fallback to Secrets
if not zrok_token:
    from kaggle_secrets import UserSecretsClient
    try:
        zrok_token = UserSecretsClient().get_secret("zrok_token")
    except:
        pass

if not zrok_token:
    print("❌ Zrok Token not found! Ensure 'sage-zrok-token' dataset is added.")
    sys.exit(1)

print(f"✅ Zrok Token loaded.")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 2: INSTALLATION & SETUP (ISOLATION FIX)
# ---------------------------------------------------------------------------
import os
import shutil
import sys

# --- FIX: Clear PYTHONPATH to prevent Kaggle globals from breaking venv ---
if "PYTHONPATH" in os.environ:
    print(f"🧹 Clearing PYTHONPATH (was: {len(os.environ['PYTHONPATH'])} chars)")
    del os.environ["PYTHONPATH"]

# 1. Install System Dependencies
!apt-get update && apt-get install -y libgl1-mesa-glx > /dev/null

# 2. Setup Workspace
WORK_DIR = "/kaggle/working/pyapi"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
    
print(f"📂 Copying PyAPI from {PYAPI_SOURCE_DIR} to {WORK_DIR}...")
try:
    shutil.copytree(PYAPI_SOURCE_DIR, WORK_DIR)
except Exception as e:
    print(f"❌ Error copying files: {e}")
    sys.exit(1)

# 3. Create Virtual Environment
VENV_PATH = os.path.join(WORK_DIR, "venv")
print(f"🔨 Creating virtual environment at: {VENV_PATH}")

# Use --without-pip to fix the Kaggle ensurepip issue
!python3 -m venv "{VENV_PATH}" --without-pip

# Define paths
VENV_BIN = os.path.join(VENV_PATH, "bin")
VENV_PYTHON = os.path.join(VENV_BIN, "python")
VENV_PIP = os.path.join(VENV_BIN, "pip")

# 4. Bootstrap PIP manually
print("🔧 Bootstrapping pip...")
!wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py
!{VENV_PYTHON} /tmp/get-pip.py > /dev/null

# 5. Install Python Requirements
print("📦 Installing Python dependencies into venv...")

req_path = os.path.join(WORK_DIR, "requirements_server_gpu.txt")
if not os.path.exists(req_path):
    print(f"❌ Error: requirements.txt not found at {req_path}")
    sys.exit(1)

# Explicitly install wrapt first to prevent build/runtime errors ---
!{VENV_PIP} install wrapt --no-cache-dir

# Install main requirements
!{VENV_PIP} install -r "{req_path}" --no-cache-dir
!{VENV_PIP} install pyngrok nest_asyncio > /dev/null

print("✅ Installation complete. Virtual environment ready.")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 2.5: UPDATE VENV, FIX GPU & DOWNLOAD MODELS
# ---------------------------------------------------------------------------
import os
import shutil
import subprocess

# 1. Define Paths
# The directory where the API code lives
WORK_DIR = "/kaggle/working/pyapi"

# The VENV binaries (Crucial!)
VENV_BIN = os.path.join(WORK_DIR, "venv", "bin")
VENV_PIP = os.path.join(VENV_BIN, "pip")
VENV_PYTHON = os.path.join(VENV_BIN, "python")

# The weights directory: Sibling of 'pyapi' folder
# Structure:
#   /kaggle/working/pyapi  (Your code)
#   /kaggle/working/rembg  (The weights, referenced by .parent.parent in service)
REMBG_DIR = "/kaggle/working/rembg"

# 2. Fix Dependencies inside VENV
print("🔧 Updating VENV dependencies for GPU support...")

# Force upgrade rembg inside venv
subprocess.run([VENV_PIP, "install", "-U", "rembg"], check=True)

# Uninstall standard onnxruntime (CPU) and ensure onnxruntime-gpu is present
# We do this to prevent the "CPUExecutionProvider" fallback
subprocess.run([VENV_PIP, "uninstall", "-y", "onnxruntime"], check=False)
subprocess.run([VENV_PIP, "uninstall", "-y", "onnxruntime-gpu"], check=False)
subprocess.run([VENV_PIP, "install", "onnxruntime-gpu"], check=True)

# 3. Setup Model Directory
if os.path.exists(REMBG_DIR):
    shutil.rmtree(REMBG_DIR)
os.makedirs(REMBG_DIR, exist_ok=True)

print(f"📂 Model directory created at: {REMBG_DIR}")

# 4. Download BiRefNet Models

# -- A. BiRefNet Lite (FP16) --
# This is the 115MB version that works on your Tablet. Great for speed.
print("⬇️ Downloading BiRefNet-Lite (FP16)...")
!wget -q --show-progress -O "{REMBG_DIR}/birefnet-general-lite.onnx" \
    "https://huggingface.co/onnx-community/BiRefNet_lite-ONNX/resolve/main/onnx/model_fp16.onnx"

# -- B. BiRefNet General (Full) --
# High quality for when speed matters less.
print("⬇️ Downloading BiRefNet-General (Full)...")
!wget -q --show-progress -O "{REMBG_DIR}/birefnet-general.onnx" \
    "https://huggingface.co/onnx-community/BiRefNet-ONNX/resolve/main/onnx/model.onnx"


# 5. Verification (Check if VENV sees the GPU)
print("\n🔍 Verifying GPU Access inside VENV:")
check_gpu_cmd = "import onnxruntime as ort; print(f'Available Providers: {ort.get_available_providers()}')"
!{VENV_PYTHON} -c "{check_gpu_cmd}"

In [ ]:
# ---------------------------------------------------------------------------
# ZROK SETUP
# ---------------------------------------------------------------------------
import os

# Download and install Zrok
if not os.path.exists("zrok"):
    print("⬇️ Downloading Zrok...")
    !wget https://github.com/openziti/zrok/releases/download/v1.1.3/zrok_1.1.3_linux_amd64.tar.gz > /dev/null
    !tar -xzf zrok_1.1.3_linux_amd64.tar.gz > /dev/null
    !chmod +x zrok

# Disable previous session if exists (cleanup)
!./zrok disable > /dev/null 2>&1

# Enable Zrok
print("🔗 Enabling Zrok...")
!./zrok enable --headless "$zrok_token"

In [ ]:
# ---------------------------------------------------------------------------
# CELL 4: RUN SERVER & TUNNEL (FINAL VENV EDITION)
# ---------------------------------------------------------------------------
import threading
import subprocess
import time
import os

SERVER_PORT = 8009
WORK_DIR = "/kaggle/working/pyapi" 
VENV_PYTHON = os.path.join(WORK_DIR, "venv", "bin", "python")

def run_pyapi_server():
    """Runs the FastAPI server using the VENV python with a clean environment"""
    
    # 1. Verify Python exists
    if not os.path.exists(VENV_PYTHON):
        print(f"❌ Critical Error: Python interpreter not found at {VENV_PYTHON}")
        return

    print(f"🚀 Starting PyAPI Server on port {SERVER_PORT} using venv...")
    
    # 2. Construct Command
    # We call the VENV python directly. 
    cmd = [
        VENV_PYTHON, "-m", "uvicorn", "main:app", 
        "--host", "0.0.0.0", 
        "--port", str(SERVER_PORT)
    ]
    
    # 3. Construct Isolated Environment
    # This is crucial: we strip Kaggle's global paths to prevent "sitecustomize" errors
    env = os.environ.copy()
    
    # Point to our venv
    env["VIRTUAL_ENV"] = os.path.join(WORK_DIR, "venv")
    # Prepend venv bin to PATH
    env["PATH"] = os.path.join(WORK_DIR, "venv", "bin") + ":" + env["PATH"]
    
    # CRITICAL: Remove PYTHONPATH so Kaggle's global libs don't interfere
    if "PYTHONPATH" in env:
        del env["PYTHONPATH"]

    # 4. Start Process
    process = subprocess.Popen(
        cmd, 
        cwd=WORK_DIR,  # Run inside /kaggle/working/pyapi
        env=env,       # Pass the clean environment
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )
    
    # 5. Log Output
    print("[PyAPI] Listening for logs...")
    try:
        for line in iter(process.stdout.readline, ''):
            clean_line = line.strip()
            if clean_line:
                # Filter for important messages to keep notebook clean
                if any(x in clean_line.lower() for x in ["error", "started", "exception", "failed", "uvicorn running"]):
                    print(f"[PyAPI] {clean_line}")
    except Exception as e:
        print(f"[PyAPI] Logger Error: {e}")

def start_zrok_tunnel():
    """Creates the public tunnel"""
    time.sleep(10) # Give venv/python ample time to warm up
    print("🚇 Opening Zrok Tunnel...")
    
    if os.path.exists("./zrok"):
        subprocess.run(["chmod", "+x", "./zrok"])
    
    # Start zrok share
    process = subprocess.Popen(
        ["./zrok", "share", "public", f"localhost:{SERVER_PORT}", "--headless"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    # Wait for tunnel registration
    time.sleep(10)
    
    # Get Public URL
    status_cmd = subprocess.run(["./zrok", "overview"], capture_output=True, text=True)
    print("\n" + "="*40)
    print(" 🌐 ZROK PUBLIC ACCESS ")
    print("="*40)
    print(status_cmd.stdout)
    print("="*40)

# Start Threads
thread_server = threading.Thread(target=run_pyapi_server, daemon=True)
thread_server.start()

thread_tunnel = threading.Thread(target=start_zrok_tunnel, daemon=True)
thread_tunnel.start()

In [ ]:
import time
print("✨ System operational. Running keep-alive loop.")

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Shutting down.")